####In this exercise we study the gasoline market and look at the relation between consumption and price in the USA.
We will use yearly data on these variables from 1977 to 1999. Additionally we have data on disposable income, and
some price indices. More precisely we have
• GC: log real gasoline consumption;
• PG: log real gasoline price index;
• RI: log real disposable income;
• RPT: log real price index of public transport;
• RPN: log real price index of new cars;
• RPU: log real price index of used cars.
####We consider the following model
GC= β1 + β2PG + β3RI + ε.

In [2]:
!pip -q install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 17.9 MB/s eta 0:00:00


#Questions
##In this exercise we study the gasoline market and look at the relation between consumption and price in the USA.
We will use yearly data on these variables from 1977 to 1999. Additionally we have data on disposable income, and
some price indices. More precisely we have
• GC: log real gasoline consumption;
• PG: log real gasoline price index;
• RI: log real disposable income;
• RPT: log real price index of public transport;
• RPN: log real price index of new cars;
• RPU: log real price index of used cars.
We consider the following model
GC= β1 + β2PG + β3RI + ε.


# Durbin-Wu-Hausman test for endogeneity


In [9]:
import pandas as pd
import numpy as np
from io import StringIO
from linearmodels.iv import IV2SLS
from scipy.stats import chi2
import statsmodels.api as sm

In [10]:
data = """OBS,GC,PG,RI,RPN,RPT,RPU
1977,6.809115,0.031026,3.165322,0.149898,-0.19227,-0.10243
1978,6.822789,0,3.209507,0.150639,-0.23588,-0.15569
1979,6.725719,0.194642,3.215122,0.119312,-0.27945,-0.18729
1980,6.647199,0.393255,3.199168,0.070287,-0.17748,-0.27962
1981,6.635689,0.397735,3.207992,0.030338,-0.06007,-0.16725
1982,6.634181,0.283268,3.216510,0.009283,-0.01672,-0.08316
1983,6.733664,0.206949,3.256705,0.003008,-0.001,-0.00908
1984,6.800027,0.142395,3.324752,-0.01064,0.017176,0.079524
1985,6.854003,0.105732,3.356372,-0.01404,0.026595,0.055143
1986,6.999965,-0.16316,3.393412,0.009083,0.065337,-0.00733
1987,6.998295,-0.17147,3.416196,0.008764,0.063933,-0.00441
1988,7.042825,-0.20472,3.456927,-0.0119,0.041397,-0.00254
1989,7.055541,-0.15684,3.477835,-0.03948,0.043399,-0.02946
1990,7.037095,-0.07135,3.491976,-0.07711,0.087139,-0.10562
1991,7.045734,-0.12997,3.492093,-0.08341,0.089151,-0.14259
1992,7.048248,-0.16466,3.523085,-0.08863,0.076142,-0.12997
1993,7.096711,-0.20854,3.530889,-0.09427,0.144714,-0.07619
1994,7.145222,-0.23298,3.551175,-0.08591,0.148932,-0.04485
1995,7.173911,-0.23486,3.571822,-0.09203,0.143407,0.026547
1996,7.177913,-0.19735,3.588693,-0.10402,0.147848,0.000637
1997,7.195133,-0.21771,3.618350,-0.12458,0.151209,-0.06035
1998,7.288701,-0.37973,3.652362,-0.14712,0.154852,-0.07912
1999,7.301263,-0.31076,3.685151,-0.17681,0.171155,-0.09172
"""

df = pd.read_csv(StringIO(data))

print(df.head())
print("Number of observations:", len(df))

    OBS        GC        PG        RI       RPN      RPT      RPU
0  1977  6.809115  0.031026  3.165322  0.149898 -0.19227 -0.10243
1  1978  6.822789  0.000000  3.209507  0.150639 -0.23588 -0.15569
2  1979  6.725719  0.194642  3.215122  0.119312 -0.27945 -0.18729
3  1980  6.647199  0.393255  3.199168  0.070287 -0.17748 -0.27962
4  1981  6.635689  0.397735  3.207992  0.030338 -0.06007 -0.16725
Number of observations: 23


In [11]:
X_ols = sm.add_constant(df[["PG", "RI"]])
y = df["GC"]

ols_results = sm.OLS(y, X_ols).fit()

print(ols_results.summary())

                            OLS Regression Results                            
Dep. Variable:                     GC   R-squared:                       0.987
Model:                            OLS   Adj. R-squared:                  0.986
Method:                 Least Squares   F-statistic:                     773.1
Date:                Sun, 11 Oct 2026   Prob (F-statistic):           1.15e-19
Time:                        08:43:30   Log-Likelihood:                 54.059
No. Observations:                  23   AIC:                            -102.1
Df Residuals:                      20   BIC:                            -98.71
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.6724      0.226     20.670      0.0

In [12]:
iv_model = IV2SLS.from_formula(
    "GC ~ 1 + RI + [PG ~ RPT + RPN + RPU]",
    data=df
)

iv_results = iv_model.fit(cov_type="unadjusted")

print(iv_results.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                     GC   R-squared:                      0.9868
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9855
No. Observations:                  23   F-statistic:                    1712.8
Date:                Sun, Oct 11 2026   P-value (F-stat)                0.0000
Time:                        08:44:02   Distribution:                  chi2(2)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      4.8283     0.2393     20.178     0.0000      4.3593      5.2973
RI             0.6192     0.0707     8.7614     0.00

In [13]:
dwh_result = iv_results.wu_hausman()

print(dwh_result)

Wu-Hausman test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 3.0495
P-value: 0.0969
Distributed: F(1,19)


In [15]:
alpha = 0.05

if dwh_result.pval < alpha:
    print("Reject H0.")
    print("There is evidence that PG is endogenous.")
else:
    print("Fail to reject H0.")
    print("There is insufficient evidence to conclude that PG is endogenous.")

Fail to reject H0.
There is insufficient evidence to conclude that PG is endogenous.


###Use 2SLS to estimate the price elasticity (β2). Use a constant, RI, RPT, RPN, and RPU as instruments.

#PG_t= pi_1 + pi_2RI_t + pi_3RPT_t + pi_4RPN_t + pi_5RPU_t + v_t
#GC= β1 + β2PG + β3RI + ε.

In [16]:
# 2SLS estimation

model_2sls = IV2SLS.from_formula("GC ~ 1 + RI + [PG ~ RPT + RPN + RPU]",data=df)

results_2sls = model_2sls.fit(cov_type="unadjusted")

print(results_2sls.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                     GC   R-squared:                      0.9868
Estimator:                    IV-2SLS   Adj. R-squared:                 0.9855
No. Observations:                  23   F-statistic:                    1712.8
Date:                Sun, Oct 11 2026   P-value (F-stat)                0.0000
Time:                        08:52:29   Distribution:                  chi2(2)
Cov. Estimator:            unadjusted                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      4.8283     0.2393     20.178     0.0000      4.3593      5.2973
RI             0.6192     0.0707     8.7614     0.00

In [17]:
beta2 = results_2sls.params["PG"]
p_value = results_2sls.pvalues["PG"]
ci = results_2sls.conf_int().loc["PG"]

print("Estimated price elasticity (beta2):", beta2)
print("P-value:", p_value)
print("95% confidence interval:")
print(ci)

Estimated price elasticity (beta2): -0.5043577788000849
P-value: 0.0
95% confidence interval:
lower   -0.609120
upper   -0.399595
Name: PG, dtype: float64


conclusion:
### Interpretation of the 2SLS Results

The estimated price elasticity of gasoline consumption is **−0.5044**. This means that a 1% increase in the real price of gasoline is associated with an approximately **0.50% decrease in real gasoline consumption**, holding real disposable income constant. Since both variables are expressed in logarithms, the coefficient can be interpreted directly as a price elasticity.

The p-value is reported as 0.0, indicating that it is extremely small (effectively zero at the displayed precision). Therefore, we reject the null hypothesis \(H_0:\beta_2=0\) at the 5% significance level. The estimated price elasticity is statistically significant.

Furthermore, the 95% confidence interval ranges from −0.6091 to −0.3996. Since this interval does not contain zero, it provides further evidence that the price elasticity is negative and statistically significant.

**Conclusion:** The 2SLS results suggest that gasoline consumption responds negatively to changes in gasoline prices. A 1% increase in the real price of gasoline is associated with a decrease in gasoline consumption of approximately 0.50%, ceteris paribus. Since the absolute value of the estimated elasticity is less than one, gasoline demand is price-inelastic over the period studied.

##Perform a Sargan test to test whether the five instruments are correlated with ε. What do you conclude?

In [18]:
sargan = results_2sls.sargan

print("Sargan statistic:", sargan.stat)
print("Degrees of freedom:", sargan.df)
print("P-value:", sargan.pval)

alpha = 0.05

if sargan.pval < alpha:
    print("Reject H0: Evidence against the validity of the overidentifying restrictions.")
else:
    print("Fail to reject H0: No sufficient evidence against the overidentifying restrictions.")

Sargan statistic: 3.895832947315366
Degrees of freedom: 2
P-value: 0.14257081238878389
Fail to reject H0: No sufficient evidence against the overidentifying restrictions.


The p-value is 0.1426, which is greater than 0.05. Therefore, we fail to reject the null hypothesis. This means that there is insufficient statistical evidence to conclude that the overidentifying restrictions are invalid. In other words, the test does not provide sufficient evidence that the instruments are correlated with the structural error term.